# 26 — Full (K, λ) Phase Diagram at N=16

Map the synchronisation order parameter R(K, λ) across the full
coupling-strength × FIM-strength plane. Identify:
1. **Critical line** K_c(λ) separating desync from sync
2. **Tricritical point** where the transition character changes
3. **Phase boundaries** — is there a distinct FIM-dominated regime?

Also measure:
- Susceptibility χ = ∂R/∂K (peaks at transition)
- Binder cumulant U = 1 - <R⁴>/(3<R²>²) (universality class)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_gradient(phases, i):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases[i] + np.pi) % (2 * np.pi) - np.pi
    sensitivity = 1.0 / (1.0 - R**2 + 0.01)
    return (1.0 / n) * np.sin(phase_diff) * min(sensitivity, 50.0)


def simulate_full(N, K_scale, fim_lambda, dt=0.02, T=100.0, noise=0.05, seed=42):
    """Return R statistics: mean, std, R2, R4 for Binder cumulant."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    R_tail = []
    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            for i in range(N):
                dphi[i] += fim_lambda * fim_gradient(theta, i)
        theta += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
        theta %= 2 * np.pi
        if s >= n_steps * 3 // 4:
            R_tail.append(float(np.abs(np.mean(np.exp(1j * theta)))))
    R_arr = np.array(R_tail)
    return {
        "R_mean": float(np.mean(R_arr)),
        "R_std": float(np.std(R_arr)),
        "R2_mean": float(np.mean(R_arr**2)),
        "R4_mean": float(np.mean(R_arr**4)),
    }


print(f"N={N}, ready.")

In [ ]:
# Phase diagram grid
K_vals = np.array([0, 1, 2, 3, 4, 5, 6, 8, 10, 12, 14, 16, 18, 20], dtype=float)
lam_vals = np.array([0, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 10.0], dtype=float)

R_grid = np.zeros((len(lam_vals), len(K_vals)))
binder_grid = np.zeros_like(R_grid)

print(
    f"Grid: {len(K_vals)} K values x {len(lam_vals)} λ values = {len(K_vals) * len(lam_vals)} points"
)
print()

for li, lam in enumerate(lam_vals):
    row_str = f"λ={lam:5.1f}: "
    for ki, K_scale in enumerate(K_vals):
        # Average over 3 seeds
        Rs = []
        R2s = []
        R4s = []
        for seed in [42, 123, 456]:
            stats = simulate_full(N, K_scale, lam, seed=seed)
            Rs.append(stats["R_mean"])
            R2s.append(stats["R2_mean"])
            R4s.append(stats["R4_mean"])
        R_grid[li, ki] = np.mean(Rs)
        r2_avg = np.mean(R2s)
        r4_avg = np.mean(R4s)
        if r2_avg > 1e-10:
            binder_grid[li, ki] = 1 - r4_avg / (3 * r2_avg**2)
        row_str += f"{R_grid[li, ki]:.2f} "
    print(row_str)

print("\nPhase diagram computed.")

In [ ]:
# Extract critical line K_c(λ)
R_CRIT = 0.5  # transition defined at R=0.5
K_c_line = []
for li, lam in enumerate(lam_vals):
    R_row = R_grid[li]
    # Interpolate to find K where R crosses R_CRIT
    crossed = False
    for ki in range(len(K_vals) - 1):
        if R_row[ki] < R_CRIT <= R_row[ki + 1]:
            # Linear interpolation
            frac = (R_CRIT - R_row[ki]) / (R_row[ki + 1] - R_row[ki] + 1e-10)
            K_c = K_vals[ki] + frac * (K_vals[ki + 1] - K_vals[ki])
            K_c_line.append((lam, float(K_c)))
            crossed = True
            break
    if not crossed and R_row[0] >= R_CRIT:
        K_c_line.append((lam, 0.0))
    elif not crossed:
        K_c_line.append((lam, float("nan")))

print("Critical line K_c(λ) at R=0.5:")
print(f"{'λ':>6} {'K_c':>8}")
for lam, kc in K_c_line:
    print(f"{lam:6.1f} {kc:8.2f}")

# Susceptibility: max dR/dK for each λ
print("\nSusceptibility peak (max dR/dK):")
print(f"{'λ':>6} {'χ_max':>8} {'at K':>8}")
for li, lam in enumerate(lam_vals):
    dR = np.diff(R_grid[li]) / np.diff(K_vals)
    chi_max = np.max(dR)
    k_peak = K_vals[np.argmax(dR)]
    print(f"{lam:6.1f} {chi_max:8.4f} {k_peak:8.1f}")

# Binder cumulant crossing
print("\nBinder cumulant U at K=10:")
ki_10 = np.argmin(np.abs(K_vals - 10))
for li, lam in enumerate(lam_vals):
    print(f"  λ={lam:5.1f}: U={binder_grid[li, ki_10]:.4f}")

In [ ]:
# Identify regimes
print("=== PHASE DIAGRAM ANALYSIS ===")
print()

# Three regimes:
# 1. Desynchronised: R < 0.3
# 2. Partially synchronised: 0.3 <= R < 0.8
# 3. Fully synchronised: R >= 0.8
print("Phase regions:")
print(f"{'lam/K':>6}", end="")
for K in K_vals:
    print(f"{K:5.0f}", end="")
print()
for li, lam in enumerate(lam_vals):
    print(f"{lam:6.1f}", end="")
    for ki in range(len(K_vals)):
        R = R_grid[li, ki]
        if R >= 0.8:
            c = "  ■ "  # sync
        elif R >= 0.3:
            c = "  ◧ "  # partial
        else:
            c = "  · "  # desync
        print(c, end="")
    print()
print("Legend: · desync (<0.3), ◧ partial (0.3-0.8), ■ sync (≥0.8)")

# Is there a pure FIM-driven sync (K=0, large λ)?
R_K0 = R_grid[:, 0]
print(f"\nPure FIM (K=0): max R = {R_K0.max():.4f} at λ={lam_vals[np.argmax(R_K0)]:.1f}")
if R_K0.max() >= 0.5:
    print("FIM ALONE can synchronise without coupling!")
else:
    print("FIM alone cannot synchronise — coupling is necessary.")

In [ ]:
# Save
output = {
    "experiment": "Phase diagram R(K, lambda) at N=16",
    "N": N,
    "K_values": K_vals.tolist(),
    "lambda_values": lam_vals.tolist(),
    "R_grid": R_grid.tolist(),
    "binder_grid": binder_grid.tolist(),
    "critical_line": K_c_line,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/phase_diagram_K_lambda_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"Results saved to {out_path}")